In [1]:
# LIbrerias
!pip install pyarrow
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, classification_report
from sklearn.ensemble import RandomForestClassifier
import joblib
import gc

### **EXPLORACIÓN DE DATOS**

In [2]:
def analisis_exploratorio_datos(df, max_cols=20, sample_size=1_000_000):
    """
    Realiza un análisis exploratorio general del DataFrame optimizado para bajo consumo de memoria.
    
    - Estadísticas descriptivas
    - Distribución de los datos (nulos y no nulos)
    - Ordena columnas por cantidad de datos no nulos (de mayor a menor)
    
    Parámetros:
    -----------
    df : pandas.DataFrame
        DataFrame a analizar.
    max_cols : int
        Número máximo de columnas a mostrar por tipo (por defecto 20).
    sample_size : int
        Tamaño máximo de muestra para calcular estadísticas (por defecto 1 millón de filas).
    """
    import pandas as pd
    import numpy as np

    print("🔍 INICIO DEL ANÁLISIS EXPLORATORIO DE DATOS\n" + "="*60)

    # 1️⃣ Dimensiones y tipos
    print(f"📏 Dimensiones: {df.shape[0]:,} filas x {df.shape[1]:,} columnas")
    print(f"🧠 Tipos de datos: {dict(df.dtypes.value_counts())}\n")

    # 2️⃣ Completitud de los datos
    print("📊 Completitud de columnas (ordenadas por datos no nulos):")
    completitud = pd.DataFrame({
        "No Nulos": df.notnull().sum(),
        "Nulos": df.isnull().sum()
    })
    completitud["% Completitud"] = 100 * completitud["No Nulos"] / len(df)
    completitud = completitud.sort_values(by="No Nulos", ascending=False)
    print(completitud.head(max_cols).round(2))

    # 3️⃣ Reducir muestra para análisis estadístico
    if len(df) > sample_size:
        df_sample = df.sample(sample_size, random_state=42)
        print(f"\n⚡ Usando muestra aleatoria de {sample_size:,} filas para estadísticas.")
    else:
        df_sample = df

    # 4️⃣ Estadísticas descriptivas numéricas
    num_cols = df_sample.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        print("\n📈 Estadísticas descriptivas numéricas:")
        stats = df_sample[num_cols].describe().T
        print(stats.head(max_cols).round(2))
    else:
        print("\n⚠️ No se encontraron columnas numéricas.")

    # 5️⃣ Variables categóricas (sin agotar RAM)
    cat_cols = df_sample.select_dtypes(exclude=[np.number]).columns
    if len(cat_cols) > 0:
        print("\n🔠 Resumen de variables categóricas:")
        resumen_cat = []
        for c in cat_cols[:max_cols]:
            vals_unicos = df_sample[c].nunique(dropna=True)
            modo = df_sample[c].mode(dropna=True)
            mas_frec = modo.iloc[0] if not modo.empty else None
            freq = df_sample[c].value_counts(dropna=True).iloc[0] if not df_sample[c].value_counts(dropna=True).empty else None
            resumen_cat.append((c, vals_unicos, mas_frec, freq))
        resumen_cat = pd.DataFrame(resumen_cat, columns=["Columna", "Valores Únicos", "Más Frecuente", "Frecuencia"])
        print(resumen_cat.to_string(index=False))
    else:
        print("\n⚠️ No se encontraron columnas categóricas.")

    # 6️⃣ Resumen final
    print("\n✅ Análisis exploratorio finalizado.")
    return completitud

In [3]:
# Ruta del archivo parquet
ruta = "C:/Users/USUARIO/Downloads/defunciones_v2.parquet"
# 1. Cargar archivo completo
df = pd.read_parquet(ruta)

In [4]:
#Analisis exploratorio
resultado_analisis_exporatorio = analisis_exploratorio_datos(df)


🔍 INICIO DEL ANÁLISIS EXPLORATORIO DE DATOS
📏 Dimensiones: 3,656,068 filas x 39 columnas
🧠 Tipos de datos: {dtype('O'): 23, Int64Dtype(): 16}

📊 Completitud de columnas (ordenadas por datos no nulos):
                     No Nulos  Nulos  % Completitud
ANO                   3656068      0          100.0
C_BAS1_DESC           3656068      0          100.0
CAUSA_666_DESC        3656068      0          100.0
C_DIR1_DESC           3656068      0          100.0
ASIS_MED_DESC         3656068      0          100.0
ASIS_MED              3656068      0          100.0
C_MUERTE_DESC         3656068      0          100.0
T_GES_AGRU_CIE_DESC   3656068      0          100.0
T_GES_DESC            3656068      0          100.0
TIPO_EMB_DESC         3656068      0          100.0
T_PARTO_DESC          3656068      0          100.0
MU_PARTO_DESC         3656068      0          100.0
CAU_HOMOL_DESC        3656068      0          100.0
CAU_HOMOL             3656068      0          100.0
MES                

In [5]:
# 0) Parámetros
SAMPLE_FRAC = 0.02   # 2% de muestra para entrenar inicialmente (ajusta según RAM)
RANDOM_STATE = 42
PARQUET_PATH = "C:/Users/USUARIO/Downloads/defunciones_completo_final.parquet"  # si necesitas recargar

In [6]:
# 1) Crear target (Opción A: si alguna fuente dijo '1' consideramos caso "confirmado")
cols_muerte = ["C_MUERTE", "C_MUERTEB", "C_MUERTEC", "C_MUERTED", "C_MUERTEE", "C_MUERTEF", "C_MUERTEG"]
# Asegúrate de que existan las columnas en df antes de ejecutar
available_muerte = [c for c in cols_muerte if c in df.columns]

df["_avoidable_proxy"] = 0
# Si cualquier columna == 1 -> 1
df.loc[df[available_muerte].eq(1).any(axis=1), "_avoidable_proxy"] = 1

print("Target balance (avoidable proxy):")
print(df["_avoidable_proxy"].value_counts(normalize=True))

Target balance (avoidable proxy):
_avoidable_proxy
1    0.993891
0    0.006109
Name: proportion, dtype: float64


In [7]:
# 2) Selección de features candidatas (ejemplo razonable; ajusta según conocimiento)
# Incluimos demográficas, contexto y variables administrativas (las que tengas)
candidate_features = [
    "SEXO", "EDAD_MADRE", "NIVEL_EDU", "EST_CIVIL", "OCUPACION",
    "NOMBRE_DEPARTAMENTO_RESIDENCIA", "NOMBRE_MUNICIPIO_RESIDENCIA",
    "COD_DPTO", "COD_MUNIC", "COD_INST", "MINUTOS", "MUERTEPORO",
    # causas codificadas - si existen
    "C_MUERTE", "C_MUERTEB", "C_MUERTEC", "C_MUERTED", "C_MUERTEE", "C_MUERTEF", "C_MUERTEG",
    # añade otras variables predictoras disponibles en tu df
]
# conservar solo las que existan
candidate_features = [c for c in candidate_features if c in df.columns]

In [8]:
# 3) Muestreo estratificado para entrenamiento (evitar MemoryError)
# si la clase positiva es muy rara podrías aumentar frac o usar upsampling
df_sample = df.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
X = df_sample[candidate_features].copy()
y = df_sample["_avoidable_proxy"].copy()

In [9]:
# 4) Separar columnas por tipo
num_cols = X.select_dtypes(include=["int64","Int64","float64","float32"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]


In [10]:
# 5) Pipeline de preprocesado
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="SinInfo")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ],
    remainder="drop",
    sparse_threshold=0
)


In [11]:
# 6) Modelo (elegimos RandomForest por robustez; luego podrías cambiar a LightGBM/XGBoost)
model = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE, class_weight="balanced")

pipeline = Pipeline(steps=[("preproc", preprocessor), ("clf", model)])


In [12]:

# 7) Train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=RANDOM_STATE)

print("Entrenando sobre muestra...", X_train.shape)
pipeline.fit(X_train, y_train)


Entrenando sobre muestra... (58496, 9)


c:\Users\USUARIO\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Pipeline(steps=[('preproc',
                 ColumnTransformer(sparse_threshold=0,
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SEXO', 'NIVEL_EDU',
                                                   'EST_CIVIL', 'C_MUERTE',
                                                   'C_MUERTEB', 'C_MUERTEC',
                                                   'C_MUERTED']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='SinInfo',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse=False))]),
                                                  ['NOMBRE_DEPARTAMENTO_RESIDENCIA',
                                                   'NOMBRE_MUNICIPIO_RESIDENCIA'])])),
                ('clf',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=200, n_jobs=-1,
                                        random_state=42))])

In [13]:

# 8) Evaluación en validación
y_pred_proba = pipeline.predict_proba(X_val)[:,1]
y_pred = pipeline.predict(X_val)
auc = roc_auc_score(y_val, y_pred_proba)
print("AUC (val):", auc)
print(classification_report(y_val, y_pred, digits=3))



AUC (val): 0.6444764065200426
              precision    recall  f1-score   support

           0      0.021     0.253     0.039        83
           1      0.995     0.934     0.964     14542

    accuracy                          0.930     14625
   macro avg      0.508     0.593     0.502     14625
weighted avg      0.990     0.930     0.958     14625



In [14]:
# 9) Guardar modelo
joblib.dump(pipeline, "modelo_mortalidad_avoidable_pipeline.joblib", compress=3)
print("Modelo guardado: modelo_mortalidad_avoidable_pipeline.joblib")



Modelo guardado: modelo_mortalidad_avoidable_pipeline.joblib


In [15]:
# 10) Scoring del dataset entero por chunks y agregación por municipio/departamento
# Creamos archivo con columnas clave y score; evitamos cargar todo en memoria.
import pyarrow.parquet as pq
pf = pq.ParquetFile(PARQUET_PATH)
batch_size = 300_000
rows_processed = 0
out_path = "C:/Users/USUARIO/Downloads/defunciones_scores.parquet"
out_df_chunks = []

# cargamos info de columnas usadas
num_cols = X.select_dtypes(include=["int64","Int64","float64","float32"]).columns.tolist()
cat_cols = [c for c in candidate_features if c not in num_cols]

for batch in pf.iter_batches(batch_size=batch_size, columns=candidate_features + ["NO", "NOMBRE_DEPARTAMENTO_RESIDENCIA", "NOMBRE_MUNICIPIO_RESIDENCIA"]):
    chunk = batch.to_pandas()
    
    # Limpieza rápida: reemplazar textos tipo 'None', 'nan', ' ' por np.nan
    chunk = chunk.replace(["None", "nan", "NaN", " ", ""], np.nan)
    
    # Forzar a numéricas las columnas numéricas (silenciosamente convierte errores a NaN)
    for c in num_cols:
        if c in chunk.columns:
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce")

    # Asegurar que las columnas categóricas existan aunque sean vacías
    for c in cat_cols:
        if c not in chunk.columns:
            chunk[c] = "SinInfo"

    # Predecir probabilidades
    probs = pipeline.predict_proba(chunk[candidate_features])[:, 1]

    # Crear miniresultado
    chunk_result = pd.DataFrame({
        "NO": chunk.get("NO"),
        "DEPARTAMENTO": chunk.get("NOMBRE_DEPARTAMENTO_RESIDENCIA"),
        "MUNICIPIO": chunk.get("NOMBRE_MUNICIPIO_RESIDENCIA"),
        "score_avoidable": probs
    })

    out_df_chunks.append(chunk_result)
    rows_processed += len(chunk)
    print(f"✅ Procesadas {rows_processed:,} filas")

# Concatenar resultados
df_scores = pd.concat(out_df_chunks, ignore_index=True)
df_scores.to_parquet(out_path, index=False, compression="snappy")
print("✅ Scoring completo guardado en:", out_path)



✅ Procesadas 300,000 filas
✅ Procesadas 600,000 filas
✅ Procesadas 900,000 filas
✅ Procesadas 1,200,000 filas
✅ Procesadas 1,500,000 filas
✅ Procesadas 1,800,000 filas
✅ Procesadas 2,100,000 filas
✅ Procesadas 2,400,000 filas
✅ Procesadas 2,700,000 filas
✅ Procesadas 3,000,000 filas
✅ Procesadas 3,300,000 filas
✅ Procesadas 3,600,000 filas
✅ Procesadas 3,900,000 filas
✅ Procesadas 4,200,000 filas
✅ Procesadas 4,500,000 filas
✅ Procesadas 4,800,000 filas
✅ Procesadas 5,100,000 filas
✅ Procesadas 5,400,000 filas
✅ Procesadas 5,700,000 filas
✅ Procesadas 6,000,000 filas
✅ Procesadas 6,300,000 filas
✅ Procesadas 6,600,000 filas
✅ Procesadas 6,900,000 filas
✅ Procesadas 7,200,000 filas
✅ Procesadas 7,500,000 filas
✅ Procesadas 7,800,000 filas
✅ Procesadas 8,100,000 filas
✅ Procesadas 8,400,000 filas
✅ Procesadas 8,700,000 filas
✅ Procesadas 8,825,031 filas
✅ Scoring completo guardado en: C:/Users/USUARIO/Downloads/defunciones_scores.parquet


In [16]:
# 11) Agregación territorial (ejemplo por municipio)
agg_mun = df_scores.groupby(["DEPARTAMENTO","MUNICIPIO"]).agg(
    n_cases = ("NO","count"),
    mean_score = ("score_avoidable","mean"),
    top_95 = ("score_avoidable", lambda x: np.percentile(x,95))
).reset_index().sort_values("mean_score", ascending=False)

agg_mun.to_csv("C:/Users/USUARIO/Downloads/agg_municipios_scores.csv", index=False)
print("Agregación por municipio guardada.")
# --------- FIN BLOQUE ----------

Agregación por municipio guardada.


In [17]:
df_scores = pd.read_parquet("C:/Users/USUARIO/Downloads/defunciones_scores.parquet")
print(df_scores.head())
print(df_scores.info())

     NO DEPARTAMENTO          MUNICIPIO  score_avoidable
0  None         META      VILLAVICENCIO         1.000000
1  None         META           CUBARRAL         1.000000
2  None         META           CUBARRAL         1.000000
3  None         META  SAN JUAN DE ARAMA         0.995056
4  None         META      FUENTE DE ORO         1.000000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8825031 entries, 0 to 8825030
Data columns (total 4 columns):
 #   Column           Dtype  
---  ------           -----  
 0   NO               object 
 1   DEPARTAMENTO     object 
 2   MUNICIPIO        object 
 3   score_avoidable  float64
dtypes: float64(1), object(3)
memory usage: 269.3+ MB
None
